# Phase 1 · Arabic NLP Foundations

> **Stack:** CAMeL Tools · pandas · pytest  
> **Corpus:** Ṣaḥīḥ al-Bukhārī (Kitāb Bad' al-Waḥy)  
> **Goal:** Build a production-grade Arabic text preprocessing pipeline  

---


## 1 · Learning Goal

### What this notebook teaches
- Normalize Arabic text across all Unicode encoding variants
- Tokenize Arabic at whitespace, word, and morpheme levels
- Extract morphological features (POS, lemma, root, segmentation)
- Identify and handle Classical Arabic (Hadith) structural patterns
- Build a reusable `HadithPreprocessor` class with a pytest test suite

### Background & key links
- [CAMeL Tools](https://camel-tools.readthedocs.io/) — NYUAD's open-source Arabic NLP toolkit (morphology, disambig, tokenization, NER, sentiment)
- [CAMeL Tools paper](https://aclanthology.org/2020.lrec-1.868/) — Obeid et al., LREC 2020
- [Unicode Arabic block](https://www.unicode.org/charts/PDF/U0600.pdf) — U+0600–U+06FF, reference for gotcha code points
- [Ṣaḥīḥ al-Bukhārī corpus](https://github.com/sunnah-com/sunnah.com) — machine-readable Hadith data

### Why it matters in real AI engineering
- **BM25 / TF-IDF retrieval** fails silently on un-normalized Arabic: `أعمال` ≠ `اعمال` ≠ `اعمالُ` — three surface forms, one semantic unit.
- **Dense embeddings** (BGE-M3, Phase 2) accept normalized text; inconsistent normalization degrades cosine similarity.
- **Chunking strategy** (Phase 3) depends on morpheme boundaries — you need to know where stems end.
- Every downstream phase inherits whatever preprocessing you do here. Garbage in → garbage vectors.


## 2 · Mental Model

### Arabic text as a layered onion

```
┌─────────────────────────────────────────────────────┐
│  Layer 4 · ROOT  (ن.و.ى)                           │
│    └── Layer 3 · STEM  (نِيَّة)                      │
│          └── Layer 2 · MORPHEMES  (بِ + ال + نِيَّة + ات) │
│                └── Layer 1 · SURFACE  (بِالنِّيَّاتِ)  │
└─────────────────────────────────────────────────────┘
```

Each layer collapses variance at different cost:

| Layer | Unique index terms | Recall | Complexity |
|---|---|---|---|
| Surface (raw) | High | Low | Zero |
| Normalized | Medium | Medium | Low |
| Morpheme-segmented | Low | High | Medium |
| Root-indexed | Very low | Very high | High |

### How this fits production RAG

```
Raw Hadith text
      │
      ▼
  [Phase 1]  HadithPreprocessor
             normalize → split isnad/matn → morpheme-tokenize
      │
      ▼
  [Phase 2]  BGE-M3 encode(normalized_tokens)
      │
      ▼
  [Phase 3]  Qdrant upsert(dense_vector + sparse_BM25)
      │
      ▼
  [Phase 4]  hybrid_search(query) → ranked passages
      │
      ▼
  [Phase 5]  Qwen3-8B generate(context + query)
```


## 3 · Background — Arabic NLP Concepts

### Core concepts

**Tashkeel (تشكيل)** — diacritical marks written above/below consonants:
- `ـَ` fatḥa (U+064E) — short /a/ vowel
- `ـِ` kasra (U+0650) — short /i/ vowel
- `ـُ` ḍamma (U+064F) — short /u/ vowel
- `ـّ` shadda (U+0651) — consonant gemination
- `ـْ` sukūn (U+0652) — no vowel / closed syllable
- `ـً ـٍ ـٌ` tanwīn — nunation (indefinite noun endings)

**Root-and-pattern morphology** — Arabic derives words by inserting a root (usually 3 consonants) into a **wazn** (pattern template):

```
Root:  ك.ت.ب  (k-t-b, 'writing/recording')

Pattern فَعَلَ  →  كَتَبَ   (kataba)  'he wrote'
Pattern فِعَال  →  كِتَاب   (kitāb)   'book'
Pattern فَاعِل  →  كَاتِب   (kātib)   'writer'
Pattern مَفْعَل  →  مَكْتَب   (maktab)  'office/desk'
Pattern مَفْعُول →  مَكْتُوب  (maktūb)  'written'
```

**Isnad vs Matn** — classical Hadith structure:

```
ISNAD (إسناد) — narrator chain
حَدَّثَنَا X عَنْ Y عَنْ Z  →  'X told us, from Y, from Z ...'
  │
  └── MATN (متن) — reported speech/action
      قَالَ رَسُولُ اللَّهِ: إِنَّمَا الأَعْمَالُ بِالنِّيَّاتِ
      'The Messenger of God said: Actions are by intentions'
```

**Why the split matters for RAG:** The isnad contains narrator names (proper nouns, largely OOV for MSA models). The matn contains the retrievable semantic content. Embedding only the matn dramatically improves retrieval precision.


## 4 · Setup — Install CAMeL Tools & Corpus



In [1]:
# Install CAMeL Tools
# Run once; comment out after first execution
import sys
!{sys.executable} -m pip install camel-tools pandas pytest --quiet


In [2]:
# Download the morphology database (required for morpheme analysis)
# This is ~200 MB and runs once
!camel_data -i morphology-db-msa-s31
# For the MLE disambiguator (used in Exercise 3):
!camel_data -i disambig-msa-r13-mle


No new packages will be installed.


Error: Invalid package name 'disambig-msa-r13-mle'


In [ ]:
# ── Sahih Bukhari sample corpus ─────────────────────────────────────────
# Kitab Bad' al-Wahy · Hadiths 1–2 (with full tashkeel)

SAMPLE_HADITHS = [
    {
        'id': 'bukhari-1',
        'raw': (
            'حَدَّثَنَا الْحُمَيْدِيُّ عَبْدُ اللَّهِ بْنُ الزُّبَيْرِ '
            'قَالَ حَدَّثَنَا سُفْيَانُ '
            'قَالَ سَمِعْتُ رَسُولَ اللَّهِ صَلَّى اللَّهُ عَلَيْهِ وَسَلَّمَ يَقُولُ '
            'إِنَّمَا الأَعْمَالُ بِالنِّيَّاتِ '
            'وَإِنَّمَا لِكُلِّ امْرِئٍ مَا نَوَى'
        )
    },
    {
        'id': 'bukhari-2',
        'raw': (
            'حَدَّثَنَا عَبْدُ اللَّهِ بْنُ يُوسُفَ '
            'قَالَ أَخْبَرَنَا مَالِكٌ عَنْ هِشَامِ بْنِ عُرْوَةَ عَنْ أَبِيهِ '
            'عَنْ عَائِشَةَ أُمِّ الْمُؤْمِنِينَ رَضِيَ اللَّهُ عَنْهَا '
            'أَنَّ الْحَارِثَ بْنَ هِشَامٍ سَأَلَ رَسُولَ اللَّهِ '
            'كَيْفَ يَأْتِيكَ الْوَحْيُ '
            'فَقَالَ أَحْيَانًا يَأْتِينِي مِثْلَ صَلْصَلَةِ الْجَرَسِ '
            'وَهُوَ أَشَدُّهُ عَلَيَّ فَيُفْصَمُ عَنِّي وَقَدْ وَعَيْتُ مَا قَالَ'
        )
    },
]

print(f'Loaded {len(SAMPLE_HADITHS)} hadiths')
for h in SAMPLE_HADITHS:
    preview = h['raw'][:60] + '...'
    print(f"  [{h['id']}] {preview}")


## 5 · Minimal Working Example — Normalize One Sentence

Before building the full pipeline, run this 10-line snippet to verify CAMeL Tools is installed correctly.


In [ ]:
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import normalize_alef_ar

sentence = 'إِنَّمَا الأَعْمَالُ بِالنِّيَّاتِ'

step1 = dediac_ar(sentence)           # strip all diacritics
step2 = normalize_alef_ar(step1)      # unify alef variants → ا
step3 = step2.replace('\u0640', '')  # strip tatweel (kashida)

print(f'Input:      {sentence}')
print(f'Normalized: {step3}')
# Expected:   انما الاعمال بالنيات



> 💡 **Tip:** If you see a `RuntimeError: Built-in database not found` in later cells, re-run the `camel_data -i` install cells above.


## Exercise 1 · Build the Normalization Pipeline

### Goal
Implement `normalize_arabic()` — a function that applies all normalization steps in the correct order and returns a clean, canonical Arabic string ready for indexing.

### Background: normalization order matters

```
CORRECT order:                    WRONG order (common mistake):
  1. NFC (Unicode composition)      dediac → normalize_alef ✓
  2. dediac_ar()                    But: NFC AFTER dediac may
  3. normalize_alef_ar()            re-introduce decomposed forms!
  4. normalize_alef_maksura_ar()    Always NFC first.
  5. remove tatweel (U+0640)
  6. collapse whitespace
```

> 🔤 **Arabic:** Arabic-Indic digits (٠١٢٣٤٥٦٧٨٩, U+0660–U+0669) appear in hadith numbering. They are NOT ASCII digits. Add an optional normalization step.



In [ ]:
import re
import unicodedata
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import (
    normalize_alef_ar,
    normalize_alef_maksura_ar,
)

_ARABIC_INDIC = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
_LIGATURES    = str.maketrans({'ﷺ': 'صلى الله عليه وسلم',   # U+FDFA
                                'ﷻ': 'جل جلاله',              # U+FDFB
                                'ﷲ': 'الله'})                # U+FDF2


def normalize_arabic(
    text: str,
    normalize_digits: bool = True,
    expand_ligatures: bool = True,
) -> str:
    """Canonical Arabic normalization pipeline.

    Parameters
    ----------
    text              : raw Arabic string (may include tashkeel, tatweel)
    normalize_digits  : convert Arabic-Indic → ASCII digits
    expand_ligatures  : expand special ligatures (ﷺ → full text)

    Returns
    -------
    Normalized string suitable for tokenization and indexing.
    """
    if not text:
        return text

    # TODO 1: Apply Unicode NFC normalization (hint: unicodedata.normalize)
    # YOUR CODE HERE

    # TODO 2: Expand Arabic ligatures if flag is set
    # YOUR CODE HERE

    # TODO 3: Strip tashkeel using dediac_ar()
    # YOUR CODE HERE

    # TODO 4: Normalize alef variants (أ إ آ ٱ → ا)
    # YOUR CODE HERE

    # TODO 5: Normalize alef maqsura (ى → ي in non-final position)
    # YOUR CODE HERE

    # TODO 6: Remove tatweel / kashida (U+0640)
    # YOUR CODE HERE

    # TODO 7: Normalize Arabic-Indic digits if flag is set
    # YOUR CODE HERE

    # TODO 8: Collapse any runs of whitespace to a single space, strip edges
    # YOUR CODE HERE

    return text


# ── Quick smoke test (do NOT change this block) ──────────────────────────
test_input  = 'إِنَّمَا الأَعْمَـالُ بِالنِّيَّـاتِ'   # has tatweel + tashkeel + أ
test_output = normalize_arabic(test_input)
print(f'Input:  {test_input}')
print(f'Output: {test_output}')
# Expected: 'انما الاعمال بالنيات'


### Questions to think about
- What happens if you apply `normalize_alef_ar()` *before* `dediac_ar()`? Try it and observe.
- The Prophet's peace-be-upon-him glyph `ﷺ` (U+FDFA) is a single code point. What does `simple_word_tokenize` do with it? What should you do for embedding?
- Is `normalize_alef_maksura_ar()` safe to apply to Classical Arabic? Think of words where `ى` vs `ي` changes meaning.


## Solution 1 · Normalization Pipeline

> Expand to see the reference solution. Try your own first!


In [ ]:
def normalize_arabic(
    text: str,
    normalize_digits: bool = True,
    expand_ligatures: bool = True,
) -> str:
    if not text:
        return text

    # 1. Unicode NFC — ensure precomposed form (hamza+alef as one codepoint)
    text = unicodedata.normalize('NFC', text)

    # 2. Expand special ligatures before any character-level ops
    if expand_ligatures:
        text = text.translate(_LIGATURES)

    # 3. Strip tashkeel (harakat, shadda, sukun, tanwin)
    text = dediac_ar(text)

    # 4. Unify alef variants: أ (0623) إ (0625) آ (0622) ٱ (0671) → ا (0627)
    text = normalize_alef_ar(text)

    # 5. Normalize alef maqsura: ى (0649) → ي (064A)
    #    Note: apply after dediac so we don't confuse ى with ي in diacritized context
    text = normalize_alef_maksura_ar(text)

    # 6. Remove tatweel / kashida (U+0640) — decorative length extension
    text = text.replace('\u0640', '')

    # 7. Arabic-Indic digits → ASCII
    if normalize_digits:
        text = text.translate(_ARABIC_INDIC)

    # 8. Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# ── Validation ───────────────────────────────────────────────────────────
cases = [
    ('إِنَّمَا الأَعْمَـالُ بِالنِّيَّـاتِ', 'انما الاعمال بالنيات'),
    ('آدَمُ',          'ادم'),                     # آ → ا
    ('يَرَى',          'يري'),                     # ى → ي (maqsura)
    ('مـحـمـد',        'محمد'),                    # tatweel stripped
    ('حديث ٢',        'حديث 2'),                  # Arabic-Indic digit
]
all_pass = True
for inp, expected in cases:
    got = normalize_arabic(inp)
    status = '✓' if got == expected else '✗'
    if got != expected:
        all_pass = False
    print(f'{status}  Input: {inp!r:30s}  Expected: {expected!r:20s}  Got: {got!r}')

print(f"\n{'All tests passed ✓' if all_pass else 'Some tests failed ✗'}")


## Exercise 2 · Three-Level Tokenization

### Goal
Implement `tokenize_levels()` — a function that returns whitespace, word-level, and morpheme-level tokenizations for an Arabic sentence. Produce a side-by-side comparison table.

### Why three levels?

| Level | Use case in RAG |
|---|---|
| Whitespace | Fast BM25 baseline, large chunks |
| Word (CAMeL) | Default production tokenization |
| Morpheme | High-recall retrieval, cross-form matching |

> 🔤 **Arabic:** Arabic clitics like `بِ` (by/with), `وَ` (and), `لِ` (for), `كَ` (like) are **prepended** to the following word as a single orthographic unit. Whitespace tokenization keeps them glued. CAMeL's `simple_word_tokenize` splits them off.



In [ ]:
import pandas as pd
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer

# Initialize the morphology analyzer once (expensive — don't do this in a loop)
_db       = MorphologyDB.builtin_db('MSA', 'analyze')
_analyzer = Analyzer(_db, 'NOAN')   # NOAN: return [] for unknown words


def get_morpheme_seg(token: str) -> str:
    """Return the d3seg morpheme segmentation for one token.

    d3seg format: 'بِ+الْ+نِيَّات+ِ'  (morpheme boundaries marked with '+')
    Falls back to the surface form if the analyzer returns no analyses.
    """
    # TODO 1: Call _analyzer.analyze(token) → list of analysis dicts
    # TODO 2: If empty, return the surface token unchanged
    # TODO 3: From the top analysis, extract 'd3seg' key
    #         If 'd3seg' is None/missing, try 'd3tok', then fall back to token
    # YOUR CODE HERE
    pass


def tokenize_levels(sentence: str) -> pd.DataFrame:
    """Tokenize at three levels and return a comparison DataFrame.

    Columns: level | count | tokens (as a space-joined string)
    Rows:    whitespace | word | morpheme
    """
    # TODO 1: Whitespace tokenization — Python str.split()
    # TODO 2: Word tokenization — simple_word_tokenize(sentence)
    # TODO 3: Morpheme tokenization — apply get_morpheme_seg to each word token
    # TODO 4: Build and return a DataFrame with columns ['level','count','tokens']
    # YOUR CODE HERE
    pass


# ── Test on the Intentions hadith matn ───────────────────────────────────
sentence = 'إِنَّمَا الأَعْمَالُ بِالنِّيَّاتِ وَإِنَّمَا لِكُلِّ امْرِئٍ مَا نَوَى'

df = tokenize_levels(sentence)
print(df.to_string(index=False))


### Questions to think about
- For the token `بِالنِّيَّاتِ`, count unique index entries at each level. How does this affect BM25 term frequency?
- The word `نَوَى` (intended/willed) has root ن.و.ي and also appears as `نِيَّة` (intention). Would L1 or L2 tokenization match them? Would L3?
- If you embed L3 morpheme tokens with BGE-M3, what happens to prefixes like `وَ` (and) that appear in almost every sentence? Think IDF.


## Solution 2 · Three-Level Tokenization



In [ ]:
def get_morpheme_seg(token: str) -> str:
    analyses = _analyzer.analyze(token)
    if not analyses:
        return token
    top = analyses[0]
    return top.get('d3seg') or top.get('d3tok') or token


def tokenize_levels(sentence: str) -> pd.DataFrame:
    L1 = sentence.split()
    L2 = simple_word_tokenize(sentence)
    L3_segs = [get_morpheme_seg(t) for t in L2]

    rows = [
        {'level': 'whitespace',  'count': len(L1), 'tokens': ' | '.join(L1)},
        {'level': 'word (CAMeL)','count': len(L2), 'tokens': ' | '.join(L2)},
        {'level': 'morpheme',    'count': len(L3_segs), 'tokens': ' | '.join(L3_segs)},
    ]
    return pd.DataFrame(rows, columns=['level', 'count', 'tokens'])


sentence = 'إِنَّمَا الأَعْمَالُ بِالنِّيَّاتِ وَإِنَّمَا لِكُلِّ امْرِئٍ مَا نَوَى'
df = tokenize_levels(sentence)
print(df.to_string(index=False))
print()

# Analysis: unique index terms per level
print('Unique terms per level:')
L1_unique = len(set(sentence.split()))
L2_unique = len(set(simple_word_tokenize(sentence)))
L3_unique = len(set(
    seg for t in simple_word_tokenize(sentence)
    for seg in get_morpheme_seg(t).split('+')
))
print(f'  Whitespace: {L1_unique}')
print(f'  Word:       {L2_unique}')
print(f'  Morpheme:   {L3_unique}  ← highest recall, smallest vocabulary')


## Exercise 3 · POS Tagging → pandas DataFrame

### Goal
POS-tag five Hadith sentences using the MLE disambiguator and produce a DataFrame with: `surface`, `lemma`, `root`, `pos`, `gloss`.

### CAMeL Tools MLE Disambiguator

The `MLEDisambiguator` uses Maximum Likelihood Estimation over a large annotated corpus to pick the most likely morphological analysis for each token **given its sentence context**.

```python
mle = MLEDisambiguator.pretrained('calima-msa-r13')
results = mle.disambiguate(tokens)   # tokens = list of strings
# results[i].analyses[0].analysis    # top analysis dict
```

> ⚠️ **Warning:** The disambiguator takes a **list of tokens** (not a single string). It needs sentence context to disambiguate. Never call it token-by-token in a loop.



In [ ]:
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.tokenizers.word import simple_word_tokenize
import pandas as pd

# Five Hadith sentences covering varied POS patterns
SENTENCES = [
    'إِنَّمَا الأَعْمَالُ بِالنِّيَّاتِ',
    'وَإِنَّمَا لِكُلِّ امْرِئٍ مَا نَوَى',
    'كَيْفَ يَأْتِيكَ الْوَحْيُ',
    'أَحْيَانًا يَأْتِينِي مِثْلَ صَلْصَلَةِ الْجَرَسِ',
    'فَمَنْ كَانَتْ هِجْرَتُهُ إِلَى اللَّهِ',
]

# TODO 1: Initialize the MLE disambiguator
#         mle = MLEDisambiguator.pretrained('calima-msa-r13')
# YOUR CODE HERE


def pos_tag_sentences(sentences: list[str]) -> pd.DataFrame:
    """POS-tag a list of Arabic sentences.

    Returns a DataFrame with columns:
      sentence_id | surface | lemma | root | pos | gloss

    For OOV tokens (no analysis), fill lemma/root/gloss with 'N/A'
    and set pos to 'UNK'.
    """
    rows = []

    for sent_id, sent in enumerate(sentences, 1):
        # TODO 2: Tokenize the sentence with simple_word_tokenize
        # TODO 3: Disambiguate ALL tokens at once (pass the full token list)
        # TODO 4: Iterate over (token, result) pairs:
        #           - If result.analyses is non-empty:
        #               top = result.analyses[0].analysis
        #               extract: lex (lemma), root, pos, gloss
        #           - Otherwise: fill with 'N/A' / 'UNK'
        # TODO 5: Append a dict row per token
        # YOUR CODE HERE
        pass

    return pd.DataFrame(rows)


df = pos_tag_sentences(SENTENCES)
pd.set_option('display.max_colwidth', 20)
print(df.to_string(index=False))


## Solution 3 · POS Tagging DataFrame



In [ ]:
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.tokenizers.word import simple_word_tokenize
import pandas as pd

mle = MLEDisambiguator.pretrained('calima-msa-r13')


def pos_tag_sentences(sentences: list[str]) -> pd.DataFrame:
    rows = []
    for sent_id, sent in enumerate(sentences, 1):
        tokens  = simple_word_tokenize(sent)
        results = mle.disambiguate(tokens)   # context-aware

        for token, result in zip(tokens, results):
            if result.analyses:
                top = result.analyses[0].analysis
                rows.append({
                    'sent_id': sent_id,
                    'surface': token,
                    'lemma':   top.get('lex',   'N/A'),
                    'root':    top.get('root',  'N/A'),
                    'pos':     top.get('pos',   'UNK'),
                    'gloss':   top.get('gloss', 'N/A'),
                })
            else:
                # Common for isnad proper nouns: الْحُمَيْدِيُّ, التَّيْمِيُّ, etc.
                rows.append({
                    'sent_id': sent_id,
                    'surface': token,
                    'lemma':   'N/A',
                    'root':    'N/A',
                    'pos':     'UNK',
                    'gloss':   'N/A',
                })

    return pd.DataFrame(rows)


df = pos_tag_sentences(SENTENCES)
pd.set_option('display.max_colwidth', 22)
print(df.to_string(index=False))
print(f"\nPOS distribution:")
print(df['pos'].value_counts().to_string())
print(f"\nOOV rate (UNK): {(df['pos'] == 'UNK').mean():.1%}")


## Mini Project · HadithPreprocessor Class

Combine everything into a production-ready class.

**Input:** raw Arabic Hadith string (may be diacritized, may contain tatweel)  
**Output:** structured dict with normalized tokens + morphological features, JSON-serializable

**Requirements:**
- Handle isnad/matn split (see `_MATN_BOUNDARY` regex below)
- Normalize with your `normalize_arabic()` function
- Tokenize at word level; morphologically analyze the matn
- Return a JSON-serializable dict (no numpy types, no CAMeL internal objects)
- Include `to_json()` method with `ensure_ascii=False`


In [ ]:
import re, json
from typing import Optional
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer

# Boundary markers that signal the start of the matn
_MATN_BOUNDARY = re.compile(
    r'(يقول|قال رسول الله|قال النبي|ان النبي|ان رسول الله|سمعت رسول)',
    re.UNICODE,
)


class HadithPreprocessor:
    """
    Production-grade Hadith preprocessing pipeline.

    Handles:
    - Unicode normalization (NFC, alef variants, tashkeel, tatweel)
    - Isnad / matn structural split
    - Word-level tokenization of the full text
    - Morphological feature extraction for matn tokens
    - JSON-serializable output
    """

    def __init__(self, db_name: str = 'MSA') -> None:
        # TODO: Initialize the MorphologyDB and Analyzer
        # YOUR CODE HERE
        pass

    @staticmethod
    def normalize(text: str) -> str:
        # TODO: Call your normalize_arabic() function from Exercise 1
        # YOUR CODE HERE
        pass

    def split_isnad_matn(self, text: str) -> tuple[str, str]:
        """Split normalized text into (isnad, matn) on first boundary match.

        Returns ('', normalized_text) if no boundary is found.
        """
        # TODO: normalize → regex search → slice
        # YOUR CODE HERE
        pass

    def _analyze_token(self, token: str) -> dict:
        """Return a JSON-serializable dict of morphological features.

        Keys: surface, lemma, root, pos, seg
        All values must be str or None (no CAMeL internal types).
        """
        # TODO: analyze → extract top analysis → build dict
        # YOUR CODE HERE
        pass

    def process(self, raw_text: str) -> dict:
        """Full pipeline. Returns:
        {
          'raw': str,
          'isnad': { 'text': str, 'tokens': list[str] },
          'matn':  { 'text': str, 'tokens': list[str], 'features': list[dict] }
        }
        """
        # TODO: wire up split_isnad_matn → tokenize → _analyze_token
        # YOUR CODE HERE
        pass

    def to_json(self, raw_text: str, indent: int = 2) -> str:
        """Process and serialize to a UTF-8 JSON string."""
        # TODO: process() → json.dumps(..., ensure_ascii=False)
        # YOUR CODE HERE
        pass


# ── Demo ─────────────────────────────────────────────────────────────────
pp = HadithPreprocessor()
result = pp.to_json(SAMPLE_HADITHS[0]['raw'])
print(result)


## Mini Project Solution · HadithPreprocessor



In [ ]:
class HadithPreprocessor:
    def __init__(self, db_name: str = 'MSA') -> None:
        db = MorphologyDB.builtin_db(db_name, 'analyze')
        self._analyzer = Analyzer(db, 'NOAN')

    @staticmethod
    def normalize(text: str) -> str:
        return normalize_arabic(text)   # from Exercise 1

    def split_isnad_matn(self, text: str) -> tuple[str, str]:
        normalized = self.normalize(text)
        match = _MATN_BOUNDARY.search(normalized)
        if match:
            return (
                normalized[:match.start()].strip(),
                normalized[match.start():].strip(),
            )
        return '', normalized

    def _analyze_token(self, token: str) -> dict:
        analyses = self._analyzer.analyze(token)
        out: dict = {
            'surface': str(token),
            'lemma':   None,
            'root':    None,
            'pos':     'UNK',
            'seg':     str(token),
        }
        if analyses:
            top = analyses[0]
            # Cast all values to str or None — no CAMeL internal types in JSON
            out.update({
                'lemma': top.get('lex')   and str(top['lex']),
                'root':  top.get('root')  and str(top['root']),
                'pos':   str(top.get('pos', 'UNK')),
                'seg':   str(top.get('d3seg') or top.get('d3tok') or token),
            })
        return out

    def process(self, raw_text: str) -> dict:
        isnad_text, matn_text = self.split_isnad_matn(raw_text)
        isnad_tokens = simple_word_tokenize(isnad_text) if isnad_text else []
        matn_tokens  = simple_word_tokenize(matn_text)
        return {
            'raw': raw_text,
            'isnad': {
                'text':   isnad_text,
                'tokens': isnad_tokens,
            },
            'matn': {
                'text':     matn_text,
                'tokens':   matn_tokens,
                'features': [self._analyze_token(t) for t in matn_tokens],
            },
        }

    def to_json(self, raw_text: str, indent: int = 2) -> str:
        return json.dumps(
            self.process(raw_text),
            ensure_ascii=False,   # keep Arabic as-is; do NOT use True
            indent=indent,
        )


pp = HadithPreprocessor()
print(pp.to_json(SAMPLE_HADITHS[0]['raw']))


## pytest Test Suite

Run this cell to write and execute the full test suite inline.


In [ ]:
# Write tests to disk then run with pytest
test_code = '''
import pytest, json
# Import from parent namespace by re-defining dependencies inline
import sys, os
sys.path.insert(0, os.getcwd())

import re, unicodedata
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import normalize_alef_ar, normalize_alef_maksura_ar
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer

_ARABIC_INDIC = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
_LIGATURES    = str.maketrans({"\\ufdf2": "الله", "\\ufdfa": "صلى الله عليه وسلم", "\\ufdfb": "جل جلاله"})
_MATN_BOUNDARY = re.compile(
    r"(يقول|قال رسول الله|قال النبي|ان النبي|ان رسول الله|سمعت رسول)",
    re.UNICODE,
)

def normalize_arabic(text, normalize_digits=True, expand_ligatures=True):
    if not text: return text
    text = unicodedata.normalize("NFC", text)
    if expand_ligatures: text = text.translate(_LIGATURES)
    text = dediac_ar(text)
    text = normalize_alef_ar(text)
    text = normalize_alef_maksura_ar(text)
    text = text.replace("\\u0640", "")
    if normalize_digits: text = text.translate(_ARABIC_INDIC)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

@pytest.fixture(scope="module")
def pp():
    from camel_tools.morphology.database import MorphologyDB
    from camel_tools.morphology.analyzer import Analyzer
    import json, re
    class HadithPreprocessor:
        def __init__(self):
            db = MorphologyDB.builtin_db("MSA", "analyze")
            self._analyzer = Analyzer(db, "NOAN")
        def normalize(self, text):
            return normalize_arabic(text)
        def split_isnad_matn(self, text):
            n = self.normalize(text)
            m = _MATN_BOUNDARY.search(n)
            return (n[:m.start()].strip(), n[m.start():].strip()) if m else ("", n)
        def _analyze_token(self, token):
            a = self._analyzer.analyze(token)
            out = {"surface": str(token), "lemma": None, "root": None, "pos": "UNK", "seg": str(token)}
            if a:
                t = a[0]
                out.update({"lemma": t.get("lex") and str(t["lex"]), "root": t.get("root") and str(t["root"]), "pos": str(t.get("pos","UNK")), "seg": str(t.get("d3seg") or t.get("d3tok") or token)})
            return out
        def process(self, raw_text):
            i, m = self.split_isnad_matn(raw_text)
            it = simple_word_tokenize(i) if i else []
            mt = simple_word_tokenize(m)
            return {"raw": raw_text, "isnad": {"text": i, "tokens": it}, "matn": {"text": m, "tokens": mt, "features": [self._analyze_token(t) for t in mt]}}
        def to_json(self, t, indent=2):
            return json.dumps(self.process(t), ensure_ascii=False, indent=indent)
    return HadithPreprocessor()

class TestNormalization:
    def test_alef_variants_unified(self, pp):
        text = "أَكَلَ إِبْرَاهِيمُ آدَمَ"
        result = pp.normalize(text)
        forbidden = {"\\u0623", "\\u0625", "\\u0622", "\\u0671"}
        assert not any(c in forbidden for c in result)
        assert "\\u0627" in result

    def test_tashkeel_fully_stripped(self, pp):
        diacritics = "\\u064b\\u064c\\u064d\\u064e\\u064f\\u0650\\u0651\\u0652\\u0670"
        text = "كَتَبَ الرَّجُلُ"
        result = pp.normalize(text)
        found = [c for c in result if c in diacritics]
        assert not found, f"Remaining diacritics {found!r} in: {result!r}"

    def test_tatweel_removed(self, pp):
        text = "مـحـمـد"
        result = pp.normalize(text)
        assert "\\u0640" not in result
        assert "محمد" in result

    def test_empty_string_safe(self, pp):
        assert pp.normalize("") == ""

    def test_isnad_matn_split_on_yaqool(self, pp):
        hadith = "حدثنا الحميدي قال سمعت رسول الله يقول انما الاعمال بالنيات"
        result = pp.process(hadith)
        assert result["isnad"]["text"] != "", "Expected non-empty isnad"
        assert result["matn"]["text"].startswith("يقول"), f"Matn must start at يقول, got: {result[\"matn\"][\"text\"][:30]!r}"
'''

with open('test_hadith_preprocessor.py', 'w', encoding='utf-8') as f:
    f.write(test_code)

print('Running pytest...')
!pytest test_hadith_preprocessor.py -v --tb=short 2>&1


## Common Mistakes & Debugging Guide

### 1. Normalization applied in the wrong order
```python
# WRONG — normalize_alef before dediac misses some composite forms
text = normalize_alef_ar(text)   # ← runs first
text = dediac_ar(text)           # ← too late

# RIGHT — always dediac first
text = dediac_ar(text)
text = normalize_alef_ar(text)
```

### 2. Forgetting `ensure_ascii=False` in json.dumps
```python
# WRONG — Arabic becomes escaped garbage
json.dumps({'text': 'بالنيات'}, ensure_ascii=True)
# Output: '{"text": "\\u0628\\u0627\\u0644\\u0646\\u064a\\u0627\\u062a"}'

# RIGHT
json.dumps({'text': 'بالنيات'}, ensure_ascii=False)
# Output: '{"text": "بالنيات"}'
```

### 3. Calling disambiguate() token-by-token (kills context)
```python
# WRONG — no sentence context; accuracy drops significantly
for token in tokens:
    result = mle.disambiguate([token])   # single-element list

# RIGHT — pass the entire sentence
results = mle.disambiguate(tokens)   # full sentence context
```

### 4. Missing database → RuntimeError
```python
# Error: RuntimeError: Built-in database not found
# Fix:
# !camel_data -i morphology-db-msa-s31
# !camel_data -i disambig-msa-r13-mle
```

### 5. Analyzer returns empty for OOV (expected for isnad)
```python
# الْحُمَيْدِيُّ (narrator name) → [] from NOAN analyzer → always handle the empty case
analyses = analyzer.analyze(token)
if not analyses:   # ← this branch fires for ~60-80% of isnad tokens
    return {'pos': 'UNK', ...}
```

### 6. RTL text renders reversed in terminal assertions
```python
# Visually confusing but bytes are correct. Use repr() or encode() for debugging:
assert result != '', f"Empty. Bytes: {result.encode('unicode_escape')}"
```


## Real AI Engineering Usage

### RAG ingestion pipeline
```python
# Phase 3 will call something like this:
def ingest_hadith_corpus(hadiths: list[dict]) -> list[dict]:
    pp = HadithPreprocessor()
    docs = []
    for h in hadiths:
        processed = pp.process(h['raw'])
        docs.append({
            'id':    h['id'],
            'text':  processed['matn']['text'],   # embed only the matn
            'meta':  {'isnad': processed['isnad']['text']},
        })
    return docs
```

### BGE-M3 embedding input (Phase 2)
```python
# Normalized matn text → encoder input
encoder_input = pp.normalize(matn_text)   # no tashkeel, no tatweel
embedding = bge_m3.encode(encoder_input)  # Phase 2
```

### Qdrant payload (Phase 3)
```python
# Morphological features stored as payload → filterable at query time
point = PointStruct(
    id=hadith_id,
    vector=embedding,
    payload={
        'matn':     processed['matn']['text'],
        'roots':    list({f['root'] for f in features if f['root']}),
        'isnad':    processed['isnad']['text'],
    }
)
```

### API response (Phase 6)
```python
# FastAPI endpoint returning processed Hadith JSON
@app.post('/process')
async def process_hadith(req: HadithRequest) -> HadithResponse:
    return json.loads(pp.to_json(req.text))   # already UTF-8 safe
```

### Evaluation (Phase 7)
```python
# Check OOV rate as a data quality metric
oov_rate = df[df['pos'] == 'UNK'].shape[0] / len(df)
# Typical Hadith corpus: 30-50% OOV for isnad, <10% for matn
# High OOV in matn → signal to check normalization
```


## Final Mini Exercise · Production Batch Processor

Process the full sample corpus through `HadithPreprocessor` and produce a report.

**Tasks:**
- Process both sample hadiths with `HadithPreprocessor.process()`
- Build a summary DataFrame: `hadith_id`, `isnad_token_count`, `matn_token_count`, `oov_rate`, `unique_roots`
- Print the top-5 most frequent POS tags across both matns
- Identify any tokens where `pos == 'UNK'` in the matn (not isnad) — these deserve investigation

> 📌 **Note:** OOV in the matn (not isnad) is usually a signal of: (a) diacritized input not pre-normalized, (b) archaic Classical Arabic verb forms, or (c) a broken normalization step.


In [ ]:
pp = HadithPreprocessor()
all_features = []
summary_rows = []

for hadith in SAMPLE_HADITHS:
    result = pp.process(hadith['raw'])
    features = result['matn']['features']

    # TODO: Compute metrics and populate summary_rows
    # YOUR CODE HERE

print('Summary:')
# pd.DataFrame(summary_rows).to_string(index=False)

print('\nTop POS tags across all matns:')
# pd.Series([f['pos'] for f in all_features]).value_counts().head(5)

print('\nOOV tokens in matn (investigate these):')
# [f['surface'] for f in all_features if f['pos'] == 'UNK']


## Key Takeaways

- **Arabic is morphologically rich.** One root produces 10+ surface forms. Surface-level matching misses them all. This is the #1 cause of Arabic RAG retrieval failures.
- **Normalization order is not optional.** NFC → dediac → normalize_alef → normalize_maqsura → strip_tatweel. Swap any two steps and you get subtle, hard-to-debug failures downstream.
- **Isnad is noise for embedding.** Narrator chains are dense proper-noun sequences, mostly OOV. Embed only the matn. Store the isnad as metadata for provenance.
- **`NOAN` over default fallback.** For Hadith text, the analyzer's default backoff produces confident-looking but wrong analyses for narrator names. `NOAN` (no-analysis) is honest about OOV.
- **`ensure_ascii=False` always.** Any JSON serialization of Arabic without this flag corrupts the text into escape sequences. Make it a team standard.
- **Three tokenization levels serve different needs.** Whitespace = fast baseline. Word (CAMeL) = production default. Morpheme = high-recall retrieval with larger index cost.
- **`HadithPreprocessor` is your Phase 1 contract.** Every downstream phase (BGE-M3, Qdrant, Qwen3) depends on its output format. Change it carefully and run the pytest suite every time.

---
> **Next:** [Phase 2 · Embeddings] — Feed normalized matn tokens into BGE-M3, explore Arabic cosine similarity, and build a first semantic search proof-of-concept.
